# 🏀 Basketball Action Recognition Using Deep Learning

This Google Colab notebook trains a **CNN + LSTM** model to classify basketball video clips into:

- Dribbling
- Passing
- Shooting
- Running

### Model
**Video → Frames → MobileNetV2 → LSTM → Action Prediction**

> Before training, place your dataset in Google Drive using:
>
> `MyDrive/Basketball_Project/dataset/{dribbling,passing,shooting,running}/`

## 1. Check TensorFlow and GPU

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

## 2. Import Libraries

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Project Configuration

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Basketball_Project/dataset"
MODEL_PATH = "/content/drive/MyDrive/Basketball_Project/basketball_action_model.keras"

CLASSES = ["dribbling", "passing", "shooting", "running"]

SEQUENCE_LENGTH = 20
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20

print("Dataset path:", DATASET_PATH)
print("Classes:", CLASSES)

## 5. Check Dataset

In [ ]:
for class_name in CLASSES:
    folder = os.path.join(DATASET_PATH, class_name)

    if not os.path.exists(folder):
        print(f"Missing folder: {folder}")
        continue

    videos = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
    ]

    print(f"{class_name}: {len(videos)} videos")

## 6. Extract Frames From a Video

Each video is converted into **20 evenly spaced frames**.  
The frames are resized to `224 × 224` for MobileNetV2.

In [ ]:
def extract_frames(video_path, sequence_length=SEQUENCE_LENGTH):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(
        0,
        max(total_frames - 1, 0),
        sequence_length
    ).astype(int)

    frames = []

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        success, frame = cap.read()

        if not success:
            cap.release()
            return None

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
        frames.append(frame)

    cap.release()

    return np.array(frames, dtype=np.float32)

## 7. Preview Frames From One Video

In [ ]:
sample_video = None

for class_name in CLASSES:
    folder = os.path.join(DATASET_PATH, class_name)

    if os.path.exists(folder):
        videos = [
            f for f in os.listdir(folder)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]

        if videos:
            sample_video = os.path.join(folder, videos[0])
            break

if sample_video is None:
    print("No videos found. Add videos to the dataset folders first.")
else:
    sample_frames = extract_frames(sample_video)

    plt.figure(figsize=(15, 6))

    for i in range(min(10, len(sample_frames))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(sample_frames[i].astype("uint8"))
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 8. MobileNetV2 Feature Extractor

MobileNetV2 is pretrained on ImageNet.  
We use it to convert every frame into a compact feature vector.

In [ ]:
cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

cnn.trainable = False

print("Feature size:", cnn.output_shape[-1])

## 9. Extract CNN Features for One Video

In [ ]:
def video_to_features(video_path):
    frames = extract_frames(video_path)

    if frames is None:
        return None

    frames = preprocess_input(frames)

    features = cnn.predict(frames, verbose=0)

    return features

## 10. Prepare the Dataset

This cell reads all videos and converts them to MobileNetV2 feature sequences.

> This can take some time depending on the number of videos.

In [ ]:
X = []
y = []

for label, class_name in enumerate(CLASSES):

    class_folder = os.path.join(DATASET_PATH, class_name)

    if not os.path.exists(class_folder):
        continue

    videos = [
        f for f in os.listdir(class_folder)
        if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
    ]

    print(f"Processing {class_name}: {len(videos)} videos")

    for index, video_name in enumerate(videos):

        video_path = os.path.join(class_folder, video_name)

        try:
            features = video_to_features(video_path)

            if features is not None:
                X.append(features)
                y.append(label)

        except Exception as e:
            print("Skipped:", video_name, "|", e)

        if (index + 1) % 10 == 0:
            print(f"  Completed {index + 1}/{len(videos)}")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("\nDataset prepared.")
print("X shape:", X.shape)
print("y shape:", y.shape)

## 11. Train / Validation / Test Split

In [ ]:
if len(X) == 0:
    raise ValueError("No usable videos were found. Check DATASET_PATH and your video files.")

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

## 12. Build the LSTM Model

The CNN features from the sequence of frames are passed to an LSTM to learn movement over time.

In [ ]:
FEATURE_SIZE = X.shape[2]

inputs = Input(shape=(SEQUENCE_LENGTH, FEATURE_SIZE))

x = LSTM(128)(inputs)
x = Dropout(0.4)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)

outputs = Dense(len(CLASSES), activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 13. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

## 14. Accuracy Graph

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

## 15. Loss Graph

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

## 16. Evaluate on Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

## 17. Classification Report

In [ ]:
probabilities = model.predict(X_test, verbose=0)
predictions = np.argmax(probabilities, axis=1)

print(
    classification_report(
        y_test,
        predictions,
        labels=list(range(len(CLASSES))),
        target_names=CLASSES,
        zero_division=0
    )
)

## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    predictions,
    labels=list(range(len(CLASSES)))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASSES
)

disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Basketball Action Recognition - Confusion Matrix")
plt.show()

## 19. Predict a New Basketball Video

Upload an unseen video from your computer when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()

test_video_path = next(iter(uploaded.keys()))

features = video_to_features(test_video_path)

if features is None:
    print("Could not process the uploaded video.")
else:
    features = np.expand_dims(features, axis=0)

    prediction = model.predict(features, verbose=0)[0]

    predicted_index = int(np.argmax(prediction))
    predicted_class = CLASSES[predicted_index]
    confidence = float(prediction[predicted_index])

    print("\n🏀 Prediction:", predicted_class.upper())
    print(f"Confidence: {confidence * 100:.2f}%")

    print("\nAll class probabilities:")
    for class_name, probability in zip(CLASSES, prediction):
        print(f"{class_name:12s}: {probability * 100:.2f}%")

## 20. Save the Final Model

In [ ]:
model.save(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

# ✅ Project Completed

### Deep Learning Techniques Used
- Transfer Learning
- MobileNetV2 CNN
- Feature Extraction
- LSTM
- Video Classification

### Classes
`Dribbling | Passing | Shooting | Running`

### Suggested GitHub Repository Name
`basketball-action-recognition-deep-learning`

Do **not** upload a large raw video dataset to GitHub. Add the notebook, README, sample results, and dataset instructions instead.